# Merton calibration & Monte Carlo — period 7-day 1-minute

**Period file:** **2023-03-09 → 2023-03-15 (1-minute bars; 5 RTH sessions)**.

> Not a 2-year regime: evaluation is **2023-03-09 → 2023-03-15** (5 RTH sessions of 1-minute bars); lookback default is **1 hour** (60 trading minutes), rolling default **minutely**. Prices are continuous RTH (overnight/weekend removed).

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** lookback default is 1 hour; rolling default is minutely, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM exercise decision on SPY American calls (risk-neutral paths from the same simulator); results in `stopping_results`.

True rolling rule: at each update, re-estimate \(\hat\mu,\hat\sigma,\hat\lambda,\hat\mu_J,\hat\kappa\) (and \(\hat\sigma_J\) for the jump-size law) from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.


## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
TICKERS = ["AAPL", "MSFT", "SPY"]
BARS_PER_DAY = 390  # regular session 09:30–16:00 ET
N_DAYS = 252 * BARS_PER_DAY  # annualize 1-minute log returns
N_STEPS = 5000  # keep the full 1-minute grid (no daily-style subsample)
JUMP_THRESH = 3.0  # flag |r| > c * 1-min σ as a jump
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "1 hour": 60,   # trading minutes (not calendar hours)
    "1 day": 390,
}
ROLLING_OPTIONS = ["minutely", "hourly"]

WINDOW_ID = "2023-03-09_to_2023-03-15"
INTRADAY_DIR = DATA / "equity" / "intraday" / "7d_1min" / WINDOW_ID
_frames = []
for _t in TICKERS:
    _p = pd.read_csv(INTRADAY_DIR / f"{_t}.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
    _frames.append(_p["Close"].rename(_t))
GAP_MIN = pd.Timedelta(minutes=2)

def _session_gap(idx) -> pd.Series:
    return pd.Series(idx, index=idx).diff() > GAP_MIN

def stitch_continuous(close: pd.Series) -> pd.Series:
    """Rebuild a gapless trading-time price: overnight/weekend returns are not applied."""
    s = close.dropna()
    r = np.log(s).diff()
    r = r.mask(_session_gap(s.index), 0.0)
    r.iloc[0] = 0.0
    return pd.Series(float(s.iloc[0]) * np.exp(r.cumsum()), index=s.index, name=s.name)

def trading_x(idx) -> np.ndarray:
    return np.arange(len(idx))

def session_starts(idx) -> np.ndarray:
    g = _session_gap(idx).fillna(False).to_numpy()
    return np.flatnonzero(g)
prices_raw = pd.concat(_frames, axis=1).sort_index()
PERIOD_START = pd.Timestamp("2023-03-09 09:30:00")
PERIOD_END = pd.Timestamp("2023-03-15 15:59:00")
prices = pd.concat(
    [stitch_continuous(prices_raw[t]).rename(t) for t in TICKERS],
    axis=1,
).sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()
# Do not treat the stitched 0-return at session joins as a 1-minute observation
for _t in TICKERS:
    _g = _session_gap(prices[_t].dropna().index)
    log_returns_all.loc[_g.reindex(log_returns_all.index, fill_value=False), _t] = np.nan


rolling = {}
cal_meta = {}

print(f"1-min sample: {prices.index.min()} → {prices.index.max()}")
print(
    f"Evaluation window: {len(period_prices)} 1-minute bars "
    f"({period_prices.index.min()} → {period_prices.index.max()}) — NOT a 2-year regime"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()


## 1. Stock price trends (7-day 1-minute, continuous RTH)

Overnight/weekend hours are dropped and prices are stitched into a continuous trading-time series. Dotted lines mark session joins.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    x = trading_x(s.index)
    ax.plot(x, s.values, color=COLORS[ticker], lw=1.4)
    for br in session_starts(s.index):
        ax.axvline(br, color="0.75", lw=0.8, ls=":")
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Close")
    ax.set_title(f"{ticker} ({role}) — continuous trading-time close (RTH only)")
axes[-1].set_xlabel("trading minute (overnight/weekend removed)")
fig.suptitle("Stock price trends — 7-day 1-minute, continuous RTH", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))

# Realized volatility from 1-minute log returns (annualized with 252 × 390)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
vol_rows = []
for ax, ticker in zip(axes, TICKERS):
    r = log_returns_all[ticker].loc[PERIOD_START:PERIOD_END].dropna()
    rv = r.rolling(30, min_periods=10).std() * np.sqrt(N_DAYS)
    ax.plot(trading_x(rv.index), rv.values, color=COLORS[ticker], lw=1.0)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("σ̂ (ann.)")
    ax.set_title(f"{ticker} ({role}) — 30-min rolling vol from 1-min returns")
    vol_rows.append({
        "ticker": ticker,
        "n_bars": int(r.shape[0]),
        "sigma_1min": float(r.std(ddof=1)),
        "sigma_ann": float(r.std(ddof=1) * np.sqrt(N_DAYS)),
        "mu_ann": float(r.mean() * N_DAYS),
    })
axes[-1].set_xlabel("trading minute")
fig.suptitle("Minute-level realized volatility — continuous RTH", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(pd.DataFrame(vol_rows).set_index("ticker").round(6))


## 2. Strike prices in this period

Listed option quotes in this repo end in **2020**, so they do not cover **2023-03-09 → 2023-03-15**.

§2 / §6 use **synthetic 2-trading-day ATM / OTM / ITM calls** on each RTH session (09:30, 11:00, 13:00, 15:00).
Benchmark = Black–Scholes European; rate = 3-month T-bill. Prices are the **continuous trading-time** series (overnight/weekend removed). SPY is the stopping underlying.


In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))
from american_lsm import make_synthetic_intraday_calls

_rates_path = DATA / "equity" / "intraday" / "7d_1min" / WINDOW_ID / "dgs3mo.csv"
_rates = (
    pd.read_csv(_rates_path, parse_dates=["observation_date"])
    .set_index("observation_date")["DGS3MO"].astype(float) / 100.0
)
_synth = {}
for ticker in TICKERS:
    _synth[ticker] = make_synthetic_intraday_calls(
        period_prices[ticker],
        log_returns_all[ticker],
        n_days=float(N_DAYS),
        period_start=PERIOD_START,
        period_end=PERIOD_END,
        rates=_rates,
        ticker=ticker,
        dte_days=2,
    )
    df = _synth[ticker]
    uniq = np.sort(df["K"].dropna().unique()) if len(df) else np.array([])
    display(Markdown(
        f"### {ticker} — {len(uniq)} synthetic strikes "
        f"(2-trading-day ATM/OTM/ITM; continuous RTH prices)"
    ))
    if len(df):
        print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
        display(df[["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]].round(4))
    else:
        print("No synthetic contracts.")
_contracts_synth_spy = _synth.get("SPY", pd.DataFrame())


## 3. Estimation formulas (Merton jump-diffusion)

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa)\, dt + \sigma\, dW_t + (e^J - 1)\, dN_t$$

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \lambda\kappa - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z + \sum_{i=1}^{N_{\Delta t}} J_i\Big)$$

with \(Z\sim N(0,1)\), \(N_{\Delta t}\sim\mathrm{Poisson}(\lambda\Delta t)\), \(J_i\sim N(\mu_J,\sigma_J^2)\).

| Parameter | Estimator (from 1-minute log returns) |
|-----------|-------------------------------------|
| \(\hat\mu\) | \(\bar r_{\text{non-jump}} \times (252\times 390)\) |
| \(\hat\sigma\) | \(s_{\text{non-jump}} \times \sqrt{252\times 390}\) |
| Jump bars | \(|r_t| > c\cdot\hat\sigma_{\text{1min}}\) with \(c=3\) |
| \(\hat\lambda\) | \(n_{\text{jumps}} / Y\) (jumps per year) |
| Jump size \(\hat\mu_J\) | mean of jump-bar returns \(J\) |
| \(\hat\sigma_J\) | std of jump-bar returns (needed for the jump-size law) |
| Compensation \(\kappa\) | \(e^{\hat\mu_J + \hat\sigma_J^2/2} - 1\) |

**Five calibrated quantities shown in §4:** \(\mu,\sigma,\lambda,\mu_J,\kappa\). \(\sigma_J\) is also rolled for simulation.

**True rolling:** at each update date, re-estimate from the lookback window ending there; those params drive the next Monte Carlo segment.


## 4. Calibration only — 7-day 1-minute

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.


In [ ]:
def estimate_merton_params(log_rets: pd.Series, jump_thresh: float = JUMP_THRESH):
    """Estimate μ, σ, λ, μ_J, σ_J, κ from a lookback window of daily log returns."""
    x = log_rets.dropna()
    n = int(x.shape[0])
    nan6 = (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, n)
    if n < 2:
        return nan6

    sigma_day = float(x.std(ddof=1))
    if not np.isfinite(sigma_day) or sigma_day <= 0:
        return nan6

    jump_mask = np.abs(x.values) > jump_thresh * sigma_day
    jumps = x.iloc[jump_mask]
    normal = x.iloc[~jump_mask]
    base = normal if int(normal.shape[0]) >= 2 else x

    mu = float(base.mean() * N_DAYS)
    sigma = float(base.std(ddof=1) * np.sqrt(N_DAYS))

    years = n / float(N_DAYS)
    n_jumps = int(jump_mask.sum())
    lam = float(n_jumps / years) if years > 0 else 0.0

    if n_jumps >= 2:
        mu_j = float(jumps.mean())
        sigma_j = float(jumps.std(ddof=1))
    elif n_jumps == 1:
        mu_j = float(jumps.iloc[0])
        sigma_j = 0.0
    else:
        mu_j = 0.0
        sigma_j = 0.0

    if not np.isfinite(sigma_j) or sigma_j < 0:
        sigma_j = 0.0

    kappa = float(np.exp(mu_j + 0.5 * sigma_j**2) - 1.0)
    return mu, sigma, lam, mu_j, sigma_j, kappa, n


def _slice_window(rets: pd.Series, end: pd.Timestamp, n_bars) -> pd.Series:
    """Last n trading-time bars ending at `end` (calendar gaps already removed)."""
    sub = rets.loc[rets.index <= pd.Timestamp(end)]
    n = int(n_bars)
    return sub.iloc[-n:] if len(sub) else sub


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    period_idx = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    if rolling_mode == "minutely":
        update_dates = period_idx
    elif rolling_mode == "hourly":
        hours = period_idx.floor("h")
        update_dates = pd.DatetimeIndex(
            [period_idx[hours == h].max() for h in hours.unique()]
        ).sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_idx[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        mu, sigma, lam, mu_j, sigma_j, kappa, n = estimate_merton_params(window)
        if n < 2 or not np.isfinite(mu):
            continue
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            "n_days": n,
            "mu": mu,
            "sigma": sigma,
            "lam": lam,
            "mu_j": mu_j,
            "sigma_j": sigma_j,
            "kappa": kappa,
        })
    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced 1-minute grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]

def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()

    def _series(col):
        return cal[col].to_numpy(dtype=float)

    mu_arr = _series("mu")
    sig_arr = _series("sigma")
    lam_arr = _series("lam")
    muj_arr = _series("mu_j")
    sj_arr = _series("sigma_j")
    kap_arr = _series("kappa")

    mu_step = np.empty(n_steps, dtype=float)
    sig_step = np.empty(n_steps, dtype=float)
    lam_step = np.empty(n_steps, dtype=float)
    muj_step = np.empty(n_steps, dtype=float)
    sj_step = np.empty(n_steps, dtype=float)
    kap_step = np.empty(n_steps, dtype=float)

    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        mu_step[i] = mu_arr[idx]
        sig_step[i] = sig_arr[idx]
        lam_step[i] = lam_arr[idx]
        muj_step[i] = muj_arr[idx]
        sj_step[i] = sj_arr[idx]
        kap_step[i] = kap_arr[idx]

    return dates, mu_step, sig_step, lam_step, muj_step, sj_step, kap_step, float(hist.iloc[0]), hist


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Five rolling-parameter graphs: μ, σ, λ, μ_J, κ."""
    panels = [
        ("mu", "μ̂ (annual)", "Estimated drift"),
        ("sigma", "σ̂ (annual)", "Estimated diffusion volatility"),
        ("lam", "λ̂ (jumps/year)", "Estimated jump intensity"),
        ("mu_j", "μ̂_J (log jump)", "Estimated jump size (mean)"),
        ("kappa", "κ̂", "Jump compensation"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
        for ax, (col, ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = np.array([
                period_prices.index.get_indexer([pd.Timestamp(d)], method="pad")[0]
                for d in r["date"]
            ])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[col], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            if col in {"mu", "mu_j", "kappa"}:
                ax.axhline(0, color="0.5", lw=0.8)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)


# Reset kernel-side UI handles so reopen + Run All cannot reuse stale widgets
plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="1 hour",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="minutely",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (graphs only)</b> — default lookback <b>1 hour</b> (60 trading minutes); default rolling <b>minutely</b>, then <b>Reestimate</b>. "
        "Shows μ̂, σ̂, λ̂, μ̂_J, κ̂. No Monte Carlo here."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated:** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown(
            "Go to **§5** and click **Start** for one MC pair per company. "
            "Jump-size vol \(\sigma_J\) is estimated in the same windows and used in simulation."
        ))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()


## 5. Monte Carlo only — one graph pair per company (7-day 1-minute)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.


In [ ]:
def simulate_merton_rolling(
    mu_step, sigma_step, lam_step, muj_step, sj_step, kap_step, S0, n_paths, seed
):
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    for i in range(n_steps):
        mu = mu_step[i]
        sigma = sigma_step[i]
        lam = lam_step[i]
        mu_j = muj_step[i]
        sigma_j = sj_step[i]
        kappa = kap_step[i]
        z = rng.standard_normal(n_paths)
        n_jumps = rng.poisson(max(lam, 0.0) * dt, size=n_paths)
        jump_sizes = np.zeros(n_paths, dtype=float)
        mask = n_jumps > 0
        if mask.any():
            jump_sizes[mask] = (
                n_jumps[mask] * mu_j
                + np.sqrt(n_jumps[mask]) * max(sigma_j, 0.0) * rng.standard_normal(int(mask.sum()))
            )
        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * sigma**2 - lam * kappa) * dt
            + sigma * np.sqrt(dt) * z
            + jump_sizes
        )
    return paths


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        (
            dates_now, mu_now, sig_now, lam_now, muj_now, sj_now, kap_now, S0_now, hist_now
        ) = param_schedule_for_steps(ticker, rolling[ticker])
        paths = simulate_merton_rolling(
            mu_now, sig_now, lam_now, muj_now, sj_now, kap_now, S0_now, n_paths, seed
        )
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            tt = trading_x(dates_now)[:_n]
            axes[0].plot(tt, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Merton Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(tt, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(tt, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("trading minute")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(np.sqrt(np.mean((p50 - _hist) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · "
            f"**ICP (25–75)** = `{100*icp:.1f}%` · "
            f"**avg band width** = `{abw:.4f}` · "
            f"RMSE(p50) = `{rmse:.4f}` | seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)


## 6. Optimal stopping (American calls — Merton)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Merton simulator** and §4 calibration on **SPY** to decide exercise vs wait for American calls.

At each day along each path:
1. Immediate payoff: $\max(S_t - K, 0)$
2. Continuation value: Longstaff–Schwartz regression on the path cloud (basis $1, S, S^2$)
3. Exercise if payoff $>$ continuation

Paths for pricing are **risk-neutral** (drift $\mu \rightarrow r$ from the option panel; vol/jumps from §4). Not the single expected path — the full Monte Carlo cloud.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping**.



In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from american_lsm import (
    lsm_american_call,
    load_spy_calls,
    make_synthetic_intraday_calls,
    params_asof,
    sample_spy_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Risk-neutral paths to expiry using §5 Merton simulator (μ → r)."""
    p = params_asof(rolling["SPY"], row.trading_date)
    if p is None:
        raise RuntimeError("No SPY calibration — run Reestimate in §4 first.")
    dte_days = int(row.dte)
    if dte_days < 2:
        raise ValueError("dte must be >= 2 trading days")
    n_steps = int(dte_days) * int(BARS_PER_DAY)
    r = float(row.r)
    S0 = float(row.S_t)
    mu_step = np.full(n_steps, r, dtype=float)
    sig_step = np.full(n_steps, float(p["sigma"]), dtype=float)
    lam_step = np.full(n_steps, float(p["lam"]), dtype=float)
    muj_step = np.full(n_steps, float(p["mu_j"]), dtype=float)
    sj_step = np.full(n_steps, float(p["sigma_j"]), dtype=float)
    kap_step = np.full(n_steps, float(p["kappa"]), dtype=float)
    return simulate_merton_rolling(
        mu_step, sig_step, lam_step, muj_step, sj_step, kap_step, S0, n_paths, seed
    )


_spy_calls_all = load_spy_calls(DATA)
_contracts = sample_spy_calls(
    _spy_calls_all, PERIOD_START, PERIOD_END, n_total=24, seed=42
)
if _contracts is None or len(_contracts) == 0:
    _rates_path = DATA / "equity" / "intraday" / "7d_1min" / WINDOW_ID / "dgs3mo.csv"
    _rates = (
        pd.read_csv(_rates_path, parse_dates=["observation_date"])
        .set_index("observation_date")["DGS3MO"].astype(float) / 100.0
    )
    _contracts = make_synthetic_intraday_calls(
        period_prices["SPY"],
        log_returns_all["SPY"],
        n_days=float(N_DAYS),
        period_start=PERIOD_START,
        period_end=PERIOD_END,
        rates=_rates,
        ticker="SPY",
        dte_days=2,
    )
stopping_results = None

display(Markdown(
    f"Using **{len(_contracts)}** SPY American calls in "
    f"{PERIOD_START.date()} → {PERIOD_END.date()} "
    f"(listed panel empty → synthetic 2-trading-day ATM/OTM/ITM vs BS European)."
))
if len(_contracts):
    display(
        _contracts[
            ["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]
        ].head(12)
    )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            if "SPY" not in rolling or len(rolling["SPY"]) == 0:
                display(Markdown("Run **Reestimate** in §4 first (need SPY calibration)."))
                return
            if _contracts is None or len(_contracts) == 0:
                display(Markdown("No SPY call contracts after synthetic fallback."))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            rows = []
            example = None
            dt = 1.0 / N_DAYS

            for i, row in enumerate(_contracts.itertuples(index=False)):
                paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                err = res.price - float(row.option_price)
                rows.append({
                    "trading_date": row.trading_date,
                    "S_t": float(row.S_t),
                    "K": float(row.K),
                    "dte": int(row.dte),
                    "r": float(row.r),
                    "market": float(row.option_price),
                    "model_price": res.price,
                    "error": err,
                    "early_ex_frac": res.early_exercise_frac,
                    "mean_ex_day": res.mean_exercise_step,
                })
                if example is None:
                    example = (row, paths, res)

            stopping_results = pd.DataFrame(rows)
            rmse = float(np.sqrt(np.mean(stopping_results["error"] ** 2)))
            mae = float(np.mean(np.abs(stopping_results["error"])))

            display(Markdown(
                f"### Merton — LSM results (SPY)\n"
                f"n_paths={n_paths} | contracts={len(stopping_results)} | "
                f"RMSE={rmse:.4f} | MAE={mae:.4f} | "
                f"mean early-exercise fraction="
                f"{stopping_results['early_ex_frac'].mean():.3f}"
            ))
            display(
                stopping_results[
                    ["trading_date", "S_t", "K", "dte", "market", "model_price",
                     "error", "early_ex_frac", "mean_ex_day"]
                ].round(4)
            )

            with plt.ioff():
                fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                ax = axes[0]
                ax.scatter(
                    stopping_results["market"], stopping_results["model_price"],
                    alpha=0.75, color=COLORS.get("SPY", "#2ca02c"),
                )
                lo = min(stopping_results["market"].min(), stopping_results["model_price"].min())
                hi = max(stopping_results["market"].max(), stopping_results["model_price"].max())
                ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                ax.set_xlabel("BS European benchmark")
                ax.set_ylabel("model LSM price")
                ax.set_title("Price: model vs BS benchmark")

                axes[1].bar(
                    ["model", "BS bench"],
                    [stopping_results["model_price"].mean(), stopping_results["market"].mean()],
                    color=[COLORS.get("SPY", "#2ca02c"), "#7f7f7f"],
                )
                axes[1].set_title("Mean option value")
                axes[1].set_ylabel("price")

                axes[2].hist(
                    stopping_results["mean_ex_day"], bins=12,
                    color=COLORS.get("SPY", "#2ca02c"), alpha=0.85, edgecolor="white",
                )
                axes[2].set_xlabel("mean exercise day (by contract)")
                axes[2].set_title("Optimal exercise timing")
                fig.suptitle(
                    f"Merton optimal stopping | {cal_meta.get('rolling_mode')} / "
                    f"{cal_meta.get('window_label')}",
                    fontsize=11, y=1.02,
                )
                fig.tight_layout()
            _show_fig(fig)

            if example is not None:
                row, paths, res = example
                j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                t_ex = int(res.exercise_steps[j])
                with plt.ioff():
                    fig2, ax = plt.subplots(figsize=(10, 3.8))
                    ax.plot(paths[j], color=COLORS.get("SPY", "#2ca02c"), lw=1.5, label="one RN path")
                    ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                    ax.scatter(
                        [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                        label=f"exercise day {t_ex}",
                    )
                    ax.set_xlabel("day")
                    ax.set_ylabel("S")
                    ax.set_title(
                        f"Example path | trade {pd.Timestamp(row.trading_date).date()} | "
                        f"dte={int(row.dte)} | model={res.price:.3f} vs mkt={float(row.option_price):.3f}"
                    )
                    ax.legend(frameon=False, loc="best")
                    fig2.tight_layout()
                _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))



## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.

